# A third opinion: LLM-based hallucination classification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrfhyL/audio_model_initial_testing/blob/main/Colab_LLMVerdict.ipynb)

`Colab_HallucinationTaxonomy.ipynb` classifies hypotheses with thresholded text statistics. Atwany
et al. (ACL Findings 2025) do the same job by asking an LLM to compare reference against hypothesis,
and report the **Hallucination Error Rate (HER)** — hallucination errors over total examples. Their
paper publishes the prompt verbatim (Figure 5, coarse-grained; Figure 6, fine-grained), so the
method is reproducible without any code from them.

This notebook runs that classifier over the same 20 000 hypotheses and asks the question the
taxonomy alone cannot answer: **does an independent method agree with it, and does it agree about
the shape across model scale?**

### This is a replication, not an adaptation

The judge is **`gpt-4o-mini`** — the model Atwany et al. actually used, at `temperature=0.0`, the
greedy decoding their §4.3 specifies for reproducibility. Nothing about the method is re-derived or
substituted: same model, same prompt, same decoding. Where this notebook deviates at all, section 3
names the deviation and the reason.

Three reasons the run is worth doing rather than a curiosity:

- **It is a genuinely different paradigm.** The taxonomy thresholds surface statistics; the LLM reads
  the pair. Where they agree, the scale trend is not an artifact of five hand-chosen thresholds.
- **Atwany et al. report the agreement structure to compare against.** Human–human 0.71,
  human–GPT 0.60, human–Gemini 0.59, GPT–Gemini 0.78 — and human–heuristic **0.00**, for a
  threshold heuristic much like ours. That last number is the one this notebook exists to test
  against our detector rather than assume away.
- **It costs about a dollar.** Batch API, `gpt-4o-mini`, 20 000 classifications. Section 4 projects
  it before anything is spent.

### Licensing: a deliberate decision, recorded here

Every other notebook in this project keeps TIMIT reference text off the wire — digests instead of
transcripts, numbers-only files, no text in printed output. **This notebook breaks that rule
knowingly**: classifying a hypothesis against its reference requires sending both to a third-party
API, and the full-grid run transmits all 1000 references. That was an explicit choice, not an
oversight, and it is recorded in `llm_provenance.json` so a later reader sees the decision rather
than inferring it.

TIMIT is LDC93S1 — licensed, not redistributable. Confirm your LDC terms permit this before running
section 5. Nothing leaves the machine until that cell executes.

## 1. Inputs and auth

| input | source |
|---|---|
| `delta_results_full.csv` | Drive — the 20 000 hypotheses |
| `delta_provenance.json` | Drive — `reference_digest`, to verify the references |
| `halluc_taxonomy.csv` | Drive — the taxonomy verdict, for the agreement analysis |
| TIMIT `TEST/**/*.TXT` | Drive — reference transcripts |
| `corpus_digests.json` | GitHub — the draw order |

Billing note, since it is easy to assume otherwise: **a Claude.ai or ChatGPT subscription does not
cover API usage.** Programmatic calls bill against API credits on the platform account, separately
from any subscription. That is part of why `gpt-4o-mini` is the right judge here — the whole grid
comes to roughly a dollar under the Batch API's 50% discount, projected in section 4 before
anything is spent.

`JUDGES` is a list. One entry (`gpt-4o-mini`) is the exact replication and what everything below
defaults to; adding a second model computes cross-model agreement as well, mirroring the
GPT-vs-Gemini figure in their Table 4. Every downstream cell handles either case.

In [ ]:
!pip -q install openai tiktoken

import collections, csv, glob, hashlib, io, itertools, json, math, os, platform, time
import numpy as np
import openai, tiktoken
from openai import OpenAI

from google.colab import drive
drive.mount("/content/drive")

# API key: Colab secret `OPENAI_API_KEY` (recommended) or a prompt
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

client = OpenAI()

DRIVE_ROOT = "/content/drive/MyDrive/NAACL"
FULL_CSV   = os.path.join(DRIVE_ROOT, "delta_results_full.csv")     # hypotheses
PROV_IN    = os.path.join(DRIVE_ROOT, "delta_provenance.json")      # reference_digest
TAX_CSV    = os.path.join(DRIVE_ROOT, "halluc_taxonomy.csv")        # taxonomy verdict
BATCH_JSON = os.path.join(DRIVE_ROOT, "llm_batches.json")           # submitted batch ids
OUT_CSV    = os.path.join(DRIVE_ROOT, "llm_verdict_per_utterance.csv")
PROV_JSON  = os.path.join(DRIVE_ROOT, "llm_provenance.json")

MODELS = ["tiny", "base", "small", "medium", "large-v3"]            # whisper ladder
PARAMS = {"tiny": "37.2M", "base": "71.8M", "small": "240.6M",
          "medium": "762.3M", "large-v3": "1541.6M"}
CONDS  = [(5, "on"), (5, "off"), (25, "on"), (25, "off")]
HEAD   = (25, "on")

JUDGES     = ["gpt-4o-mini"]     # the paper's model. append a second id for cross-model agreement
CHUNK      = 10000               # requests per batch; ceilings are 50 000 requests / 200 MB
MAX_TOKENS = 64                  # a schema-constrained label needs very few
TEMPERATURE = 0.0                # their SS4.3 greedy decoding, verbatim

## 2. Load, verify, and join

Same integrity checks the taxonomy notebook runs, for the same reason: a classification is only
meaningful against the corpus that produced it. The references are rebuilt from TIMIT and hashed
against the `reference_digest` the delta sweep pinned — if the pin mismatches, the LLM would be
comparing hypotheses against the wrong sentences and every verdict below would be noise.

The taxonomy verdict is joined here too, so section 8 can compute agreement without re-reading
anything.

In [ ]:
def sha256_file(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


for p in (FULL_CSV, PROV_IN, TAX_CSV):
    assert os.path.exists(p), f"missing input: {p}"
INPUT_DIGESTS = {os.path.basename(p): sha256_file(p) for p in (FULL_CSV, PROV_IN, TAX_CSV)}

full    = list(csv.DictReader(open(FULL_CSV, newline="")))
tax     = list(csv.DictReader(open(TAX_CSV,  newline="")))
prov_in = json.load(open(PROV_IN))

cells = collections.Counter((r["model"], int(r["offset_s"]), r["timestamps"]) for r in full)
assert set(cells) == {(m, o, t) for m in MODELS for o, t in CONDS}, "grid has holes"
assert set(cells.values()) == {1000}, sorted(set(cells.values()))
assert sum("<|" in r["text"] for r in full) == 0, "special tokens leaked into text"

TAX = {(r["model"], r["path"], int(r["offset_s"]), r["timestamps"]): r for r in tax}
assert len(TAX) == len(full), (len(TAX), len(full))

# --- references, verified against the delta sweep's pin --------------------------------
cands = [d for d in glob.glob(os.path.join(DRIVE_ROOT, "timit", "**", "TEST"), recursive=True)
         if os.path.isdir(os.path.join(d, "DR1"))]
assert cands, f"no TIMIT TEST/DR1 found under {DRIVE_ROOT}/timit"
TIMIT_TEST = sorted(cands, key=len)[0]

if not os.path.exists("corpus_digests.json"):
    !wget -q https://raw.githubusercontent.com/AgrfhyL/audio_model_initial_testing/main/corpus_digests.json
FILES = json.load(open("corpus_digests.json"))["files"][0:1000]

!pip -q install openai-whisper
from whisper.normalizers import EnglishTextNormalizer          # noqa: E402
normalizer = EnglishTextNormalizer()


def load_reference(wav_path):
    """TIMIT .TXT is `<start_sample> <end_sample> <sentence>` -- the integers are not words."""
    with open(wav_path[:-4] + ".TXT") as f:
        return f.read().strip().split(None, 2)[2]


REF     = {r["path"]: normalizer(load_reference(os.path.join(TIMIT_TEST, r["path"])))
           for r in FILES}
rebuilt = sha256_bytes("\n".join(f"{r['path']}\t{REF[r['path']]}" for r in FILES).encode())

pinned = None
for block in prov_in.values():
    if isinstance(block, dict) and "reference_digest" in block:
        pinned = block["reference_digest"]
        break
assert pinned is not None and rebuilt == pinned, (
    "reference digest mismatch (or absent): this TIMIT copy is not the corpus experiment A "
    "decoded, so every verdict below would score against the wrong sentences")

print(f"{len(full)} hypotheses | {len(REF)} references | digest matches the delta sweep's pin")
for k, v in INPUT_DIGESTS.items():
    print(f"  {k:<28} sha256 {v[:16]}...")

## 3. The prompt, and one deliberate deviation

The instruction text below is Atwany et al.'s Figure 5 coarse-grained prompt, transcribed verbatim
including all six examples, sent to the model they used at the temperature they specify. The model,
the prompt, and the decoding all match the paper.

**One deviation: structured outputs replace "produce only the classification."** The paper
constrains the output by asking for it in prose; `response_format` with a `json_schema` in strict
mode constrains it by construction. This removes the parse step entirely and forecloses the two
failure modes a prose instruction cannot — a preamble before the label, and a label spelled
differently from the three canonical strings. It is a strict improvement on the method rather than a
change to it, and their own §A.4.1 says they were doing this by prompt design anyway ("restricts
GPT-4o's output to only the final classification decision").

The instructions go in the `system` message and the pair in the `user` message. The paper presents
one block ending with an `Input:` template; the rendered text and its order are unchanged.

The fine-grained prompt (their Figure 6, five categories) is transcribed too but not run by default
— `GRAIN` selects it. Coarse is the default because the agreement analysis in section 8 needs a
binary hallucination/not verdict to compare against the taxonomy.

In [ ]:
COARSE_LABELS = ["Hallucination Error", "Non-Hallucination Error", "No Error"]
FINE_LABELS   = ["Phonetic Error", "Oscillation Error", "Hallucination Error",
                 "Language Error", "No Error"]

COARSE_SYSTEM = """You are a classifier trained to detect and categorize specific transcription \
errors produced by a speech recognition system. The possible categories are:
1. Hallucination Error: The output contains fabricated, contradictory, or invented information \
that is not supported by the ground truth. This includes: - Fabricated Content: Words or phrases \
entirely absent in the ground truth. - Meaningful Contradictions: Significant changes in the \
meaning from the ground truth. - Invented Context: Introduction of details or context not present \
in the ground truth. - Note: These errors involve fabrication of new information or significant \
distortion of meaning, beyond grammatical or structural mistakes.
2. Non-Hallucination Error: Errors that do not involve fabrication or significant contradictions \
of the ground truth. These include: - Phonetic Errors: Substitutions of phonetically similar words \
or minor pronunciation differences. - Structural or Language Errors: Grammatical, syntactic, or \
structural issues that make the text incoherent or incorrect (e.g., incorrect verb tenses, \
subject-verb agreement problems, omissions, or insertions). - Oscillation Errors: Repetitive, \
nonsensical patterns or sounds that do not convey linguistic meaning (e.g., "ay ay ay ay"). \
- Other Non-Hallucination Errors: Errors that do not fit the above subcategories but are not \
hallucinations.
3. No Error: The generated output conveys the same meaning as the ground truth, even if the \
phrasing, grammar, or structure differs. Minor differences in wording, phrasing, or grammar that \
do not alter the intended meaning are acceptable.

Input Format:
Ground Truth: The original, accurate text provided.
Generated Output: The text produced by the speech recognition system.

Output Format: Classify the input text pairs into one of the following:
Non-Hallucination Error
Hallucination Error
No Error

Examples:
Example 1:
Ground Truth: "A millimeter roughly equals one twenty-fifth of an inch."
Generated Output: "Miller made her roughly one twenty-fifths of an inch."
Output: Non-Hallucination Error

Example 2:
Ground Truth: "Indeed, ah!"
Generated Output: "Ay ay indeed ay ay ay ay ay ay."
Output: Non-Hallucination Error

Example 3:
Ground Truth: "Captain Lake did not look at all like a London dandy now."
Generated Output: "Will you let Annabel ask her if she sees what it is you hold in your arms \
again?"
Output: Hallucination Error

Example 4:
Ground Truth: "The patient was advised to take paracetamol for fever and rest for two days."
Generated Output: "The patient was advised to take amoxicillin for fever and undergo surgery \
immediately."
Output: Hallucination Error

Example 5:
Ground Truth: "I need to book a flight to New York."
Generated Output: "I need to book ticket to New York."
Output: No Error

Example 6:
Ground Truth: "She went to the store yesterday."
Generated Output: "She went to the shop yesterday."
Output: No Error

Instruction: You must produce only the classification as the output. Do not include explanations, \
reasoning, or additional information."""

FINE_SYSTEM = """You are a classifier trained to detect and categorize specific transcription \
errors produced by a speech recognition system. The possible categories are:
1. Phonetic Error: The output contains substitutions of phonetically similar words that do not \
match the ground truth and do not introduce broader grammatical or structural issues.
2. Oscillation Error: The output includes repetitive, nonsensical patterns or sounds that do not \
convey linguistic meaning (e.g., "ay ay ay ay").
3. Hallucination Error: The output contains fabricated, contradictory, or invented information \
that is not supported by the ground truth.
4. Language Error: The output includes grammatical, syntactic, or structural issues that make the \
text incoherent or linguistically incorrect.
5. No Error: The generated output conveys the same meaning as the ground truth, even if the \
phrasing, grammar, or structure differs.

Instruction: You must produce only the classification as the output. Do not include explanations, \
reasoning, or additional information."""

GRAIN  = "coarse"                                   # "coarse" (default) or "fine"
SYSTEM = COARSE_SYSTEM if GRAIN == "coarse" else FINE_SYSTEM
LABELS = COARSE_LABELS if GRAIN == "coarse" else FINE_LABELS

# strict mode requires additionalProperties:false and every property listed in required
SCHEMA = {"type": "object",
          "properties": {"label": {"type": "string", "enum": LABELS}},
          "required": ["label"], "additionalProperties": False}
RESPONSE_FORMAT = {"type": "json_schema",
                   "json_schema": {"name": "verdict", "strict": True, "schema": SCHEMA}}


def user_turn(ref, hyp):
    return f'Ground Truth: "{ref}"\nGenerated Output: "{hyp}"'


def body_for(judge, ref, hyp):
    """One /v1/chat/completions body. Identical across judges -- unlike the Anthropic surface,
    every OpenAI chat model here takes the same temperature and response_format fields."""
    return {
        "model": judge,
        "messages": [{"role": "system", "content": SYSTEM},
                     {"role": "user",   "content": user_turn(ref, hyp)}],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "response_format": RESPONSE_FORMAT,
    }


print(f"grain={GRAIN} | {len(LABELS)} labels | system block {len(SYSTEM)} chars")
print(f"judges: {', '.join(JUDGES)} at temperature={TEMPERATURE}")
print("labels:", ", ".join(LABELS))

## 4. Measure before spending

`gpt-4o-mini` bills at **\$0.150 per 1M input / \$0.600 per 1M output**, halved by the Batch API.
Cached input is \$0.075/1M, but OpenAI's caching is automatic and only engages on prefixes of
**1024 tokens or more** — the cell below measures the instruction block with `tiktoken` and reports
whether it clears that bar. It very likely does not, and the projection assumes it does not.

Worth knowing while reading their paper: Atwany et al.'s App. A.4.1 rate card quotes
"\$0.075 per million input tokens", which is the *cached* rate, alongside "with most input tokens
cached". Their \$78-per-million-segments figure therefore assumes a near-perfect cache hit rate. The
projection below uses the base rate instead, so it should read slightly higher per row than theirs —
that is the honest direction to be wrong in.

Padding the prompt past 1024 tokens would halve the input line, and is deliberately **not** done:
the saving is a fraction of a dollar and it would mean no longer running the paper's prompt.

In [ ]:
PRICE = {"gpt-4o-mini": {"in": 0.150, "out": 0.600}}      # USD / 1M tokens, uncached
BATCH_DISCOUNT, CACHE_MIN = 0.50, 1024

enc = tiktoken.encoding_for_model("gpt-4o-mini")
_ref, _hyp = REF[FILES[0]["path"]], normalizer(full[0]["text"])
sys_tok = len(enc.encode(SYSTEM))
var_tok = len(enc.encode(user_turn(_ref, _hyp)))
out_tok, N = 12, len(full)

print(f"system block {sys_tok} tokens | per-row variable ~{var_tok} | ~{out_tok} out | N = {N}")
print(f"automatic caching engages at {CACHE_MIN} tokens: "
      f"{'YES, prefix qualifies' if sys_tok >= CACHE_MIN else 'no, prefix is below the bar'}\n")

hdr = f"{'judge':>16} {'input $':>9} {'output $':>9} {'sync $':>9} {'batch total $':>14}"
print(hdr + "\n" + "-" * len(hdr))
TOTAL = 0.0
for j in JUDGES:
    p = PRICE.get(j)
    if p is None:
        print(f"{j:>16} {'(no rate card entry -- add it to PRICE to project)':>50}")
        continue
    cin  = N * (sys_tok + var_tok) / 1e6 * p["in"]
    cout = N * out_tok / 1e6 * p["out"]
    tot  = (cin + cout) * BATCH_DISCOUNT
    TOTAL += tot
    print(f"{j:>16} {cin:>9.2f} {cout:>9.2f} {cin + cout:>9.2f} {tot:>14.2f}")
print("-" * len(hdr))
print(f"{'total':>16} {'':>9} {'':>9} {'':>9} {TOTAL:>14.2f}")
print(f"\nAtwany et al. scale to ~$1.56 for {N} rows at their (cached-rate) card.")
print("A projection far outside this order of magnitude means the request shape is wrong.")

## 5. Submit — the cell that sends reference text off this machine

**This is the licensing boundary.** Running it transmits all 1000 TIMIT reference transcripts, plus
every hypothesis, to the OpenAI API. Nothing above this point has left the runtime.

The Batch API takes a JSONL file rather than an in-memory request list: one line per call, each
carrying `custom_id`, `method`, `url`, and `body`. The file is uploaded with `purpose="batch"`, then
referenced by id. Ceilings are 50 000 requests and 200 MB per batch — 20 000 rows at this prompt
size is roughly 66 MB, so `CHUNK = 10000` stays comfortably inside both while giving two resume
points.

Every batch id is written to `llm_batches.json` on Drive *before* the next chunk goes out. Batches
are durable server-side, so a disconnected runtime costs nothing: re-running this cell re-reads the
id file and submits only what is missing, and section 6 collects results from ids alone. `custom_id`
is the row's index into `full`, which is stable because `full` comes from a digest-verified file.

In [ ]:
CONFIRM = False        # set True to send reference + hypothesis text to the OpenAI API

assert CONFIRM, ("set CONFIRM = True to proceed. This transmits TIMIT reference transcripts "
                 "(LDC93S1, licensed) to a third-party API; confirm your LDC terms permit it.")

submitted = json.load(open(BATCH_JSON)) if os.path.exists(BATCH_JSON) else {}

for judge in JUDGES:
    done = submitted.setdefault(judge, {})
    for start in range(0, len(full), CHUNK):
        key = str(start)
        if key in done:
            print(f"{judge} rows {start}-{start+CHUNK}: already submitted ({done[key]})")
            continue

        path = f"/content/batch_{judge}_{start}.jsonl"
        with open(path, "w") as f:
            for i in range(start, min(start + CHUNK, len(full))):
                r = full[i]
                f.write(json.dumps({
                    "custom_id": f"r{i}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": body_for(judge, REF[r["path"]], normalizer(r["text"])),
                }) + "\n")
        mb = os.path.getsize(path) / 1e6
        assert mb < 200, f"{mb:.0f} MB exceeds the 200 MB batch ceiling; lower CHUNK"

        up    = client.files.create(file=open(path, "rb"), purpose="batch")
        batch = client.batches.create(input_file_id=up.id,
                                      endpoint="/v1/chat/completions",
                                      completion_window="24h",
                                      metadata={"description": f"HER {judge} rows {start}"})
        done[key] = batch.id
        with open(BATCH_JSON, "w") as f:          # persist before the next chunk
            json.dump(submitted, f, indent=1)
        print(f"{judge} rows {start}-{start+CHUNK}: {batch.id}  ({mb:.0f} MB uploaded)")

print(f"\n{sum(len(v) for v in submitted.values())} batches on record -> {BATCH_JSON}")

## 6. Poll and collect

Batches usually finish well inside the 24-hour window. This cell polls each to a terminal status,
then downloads and parses the output file.

Two details the Batch API makes easy to get wrong. **Failed requests do not appear in the output
file at all** — they go to a separate `error_file_id`, so a batch that silently dropped 300 rows
still yields a clean-looking output file. Both files are read below and the row count is asserted.
And **results are not in submission order**, so everything is keyed by `custom_id`.

`cached_tokens` is reported from the returned usage rather than assumed — if section 4 said the
prefix was below the caching bar, this should read zero, and a non-zero value means the projection
was pessimistic rather than wrong.

In [ ]:
TERMINAL = {"completed", "failed", "expired", "cancelled"}
submitted = json.load(open(BATCH_JSON))
VERDICT, usage_tot, errors = {}, collections.Counter(), []

for judge, chunks in submitted.items():
    for key, bid in sorted(chunks.items(), key=lambda kv: int(kv[0])):
        while True:
            b = client.batches.retrieve(bid)
            if b.status in TERMINAL:
                break
            print(f"  {judge} {bid}: {b.status} "
                  f"({b.request_counts.completed}/{b.request_counts.total})", flush=True)
            time.sleep(60)
        assert b.status == "completed", f"{bid} ended {b.status}"

        for line in client.files.content(b.output_file_id).text.splitlines():
            if not line.strip():
                continue
            o = json.loads(line)
            i = int(o["custom_id"][1:])
            body = o["response"]["body"]
            VERDICT[(judge, i)] = json.loads(body["choices"][0]["message"]["content"])["label"]
            u = body.get("usage", {})
            usage_tot[(judge, "in")]     += u.get("prompt_tokens", 0)
            usage_tot[(judge, "out")]    += u.get("completion_tokens", 0)
            usage_tot[(judge, "cached")] += (u.get("prompt_tokens_details") or {}).get(
                "cached_tokens", 0)

        if getattr(b, "error_file_id", None):     # failures live in a separate file
            for line in client.files.content(b.error_file_id).text.splitlines():
                if line.strip():
                    errors.append(json.loads(line))
        print(f"{judge} chunk {key}: collected ({bid})")

assert not errors, f"{len(errors)} failed requests, e.g. {errors[0]}"
for judge in JUDGES:
    n = sum(1 for (j, _) in VERDICT if j == judge)
    assert n == len(full), f"{judge}: {n} verdicts for {len(full)} rows"
assert set(VERDICT.values()) <= set(LABELS), set(VERDICT.values()) - set(LABELS)

print(f"\n{len(VERDICT)} verdicts, no failures\n")
hdr = f"{'judge':>16} {'prompt tok':>11} {'cached':>9} {'completion':>11} {'cached %':>9}"
print(hdr + "\n" + "-" * len(hdr))
for j in JUDGES:
    i_, c_, o_ = (usage_tot[(j, k)] for k in ("in", "cached", "out"))
    print(f"{j:>16} {i_:>11} {c_:>9} {o_:>11} {100*c_/max(1, i_):>8.1f}%")

## 7. Hallucination Error Rate

HER is Atwany et al.'s metric: hallucination errors over total examples in the cell. It is directly
comparable to the taxonomy's hallucination rate — same denominator, same 1000 clips — which is what
makes section 8's agreement analysis possible at all.

Wilson intervals again, and the speaker-clustered bootstrap on the headline condition, for the same
reason as everywhere else in this project: 168 speakers contribute 5–6 clips each, so utterances are
not independent.

In [ ]:
def wilson(k, n, z=1.959963985):
    if n == 0:
        return float("nan"), 0.0, 1.0
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


def cluster_ci(flags, speakers, n_boot=10000, alpha=0.05, seed=0):
    f, g = np.asarray(flags, float), np.asarray(speakers)
    pool = [np.flatnonzero(g == u) for u in np.unique(g)]
    K, rng = len(pool), np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        boots[b] = f[np.concatenate([pool[j] for j in rng.integers(0, K, K)])].mean()
    return float(np.quantile(boots, alpha / 2)), float(np.quantile(boots, 1 - alpha / 2))


rows = []
for i, r in enumerate(full):
    t = TAX[(r["model"], r["path"], int(r["offset_s"]), r["timestamps"])]
    row = {"model": r["model"], "path": r["path"], "speaker": t["speaker"],
           "region": t["region"], "offset_s": int(r["offset_s"]),
           "timestamps": r["timestamps"], "taxonomy": t["category"],
           "taxonomy_halluc": int(t["halluc"])}
    for j in JUDGES:
        row[f"verdict_{j}"] = VERDICT[(j, i)]
        row[f"halluc_{j}"]  = int(VERDICT[(j, i)] == "Hallucination Error")
    rows.append(row)

BY = collections.defaultdict(list)
for r in rows:
    BY[(r["model"], r["offset_s"], r["timestamps"])].append(r)

SUMMARY = {}
for src in JUDGES + ["taxonomy"]:
    col = "taxonomy_halluc" if src == "taxonomy" else f"halluc_{src}"
    label = "taxonomy hallucination rate (%)" if src == "taxonomy" else f"HER (%) — judge {src}"
    print(f"\n{label}   [Wilson 95% CI]\n")
    hdr = f"{'model':>9} {'params':>8} " + " ".join(f"{f'{o} s / ts {t}':>21}" for o, t in CONDS)
    print(hdr + "\n" + "-" * len(hdr))
    for m in MODELS:
        out = []
        for o, t in CONDS:
            rs = BY[(m, o, t)]
            p, lo, hi = wilson(sum(r[col] for r in rs), len(rs))
            SUMMARY[f"{src}|{m}|{o}|{t}"] = {"n": len(rs), "her": p,
                                             "wilson_lo": lo, "wilson_hi": hi}
            out.append(f"{100*p:5.1f} [{100*lo:4.1f},{100*hi:4.1f}]")
        print(f"{m:>9} {PARAMS[m]:>8} " + " ".join(f"{c:>21}" for c in out))

def ci(lo, hi):
    """Format an interval without nesting quotes inside an f-string (works on Python 3.9+)."""
    return "[" + f"{100*lo:5.2f}" + ", " + f"{100*hi:5.2f}" + "]"


print(f"\nspeaker-clustered cross-check at {HEAD[0]} s / ts {HEAD[1]}\n")
hdr = f"{'source':>16} {'model':>10} {'wilson 95%':>18} {'speaker-clustered':>20}"
print(hdr + "\n" + "-" * len(hdr))
for src in JUDGES + ["taxonomy"]:
    col = "taxonomy_halluc" if src == "taxonomy" else f"halluc_{src}"
    for m in MODELS:
        rs = BY[(m, *HEAD)]
        s  = SUMMARY[f"{src}|{m}|{HEAD[0]}|{HEAD[1]}"]
        lo, hi = cluster_ci([r[col] for r in rs], [r["speaker"] for r in rs])
        s["cluster_lo"], s["cluster_hi"] = lo, hi
        print(f"{src:>16} {m:>10} {ci(s['wilson_lo'], s['wilson_hi']):>18} "
              f"{ci(lo, hi):>20}")

## 8. Agreement — the number this notebook exists to produce

Atwany et al. report raw agreement between evaluation methods: human–human 0.71, human–GPT 0.60,
human–Gemini 0.59, GPT–Gemini 0.78, and human–heuristic **0.00** against a threshold heuristic
(cosine similarity, WER, and perplexity cuts) not unlike ours.

That 0.00 is the reason to run this. If our taxonomy agrees with `gpt-4o-mini` at anything
approaching their human–GPT figure, the "heuristics don't work" result does not transfer to this
detector, and we can say so with a number instead of an argument. If it agrees at near zero, that is
a finding about our detector far better discovered here than in review.

Two quantities, because agreement alone is ambiguous at low base rates: **raw agreement** over all
20 000 rows (dominated by the ~98% both methods call clean) and **Cohen's kappa**, which corrects
for chance and is the honest figure when one class is rare. Positive-class Jaccard is printed
alongside, since that is where the two methods actually have to decide something.

In [ ]:
def agree(a, b):
    """Raw agreement, Cohen's kappa, and Jaccard over the positive class."""
    a, b = np.asarray(a, int), np.asarray(b, int)
    po = float((a == b).mean())
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    both, either = int((a & b).sum()), int((a | b).sum())
    return po, kappa, (both / either if either else float("nan")), both, either


COL = {j: f"halluc_{j}" for j in JUDGES}
COL["taxonomy"] = "taxonomy_halluc"
PAIRS = list(itertools.combinations(["taxonomy"] + JUDGES, 2))   # every pair, any judge count

print(f"agreement over all {len(rows)} rows\n")
hdr = f"{'pair':>44} {'raw':>7} {'kappa':>7} {'jaccard':>8} {'both':>6} {'either':>7}"
print(hdr + "\n" + "-" * len(hdr))
AGREE = {}
for x, y in PAIRS:
    po, k, jac, both, either = agree([r[COL[x]] for r in rows], [r[COL[y]] for r in rows])
    AGREE[f"{x} vs {y}"] = {"raw": po, "kappa": k, "jaccard": jac,
                            "both": both, "either": either}
    print(f"{f'{x} vs {y}':>44} {po:>7.3f} {k:>7.3f} {jac:>8.3f} {both:>6} {either:>7}")

print("\nAtwany et al. Table 4 for reference:")
print("  human-human 0.71 | human-GPT 0.60 | human-Gemini 0.59 | GPT-Gemini 0.78")
print("  human-heuristic 0.00 | GPT-heuristic 0.10 | heuristic-Gemini 0.14")
print("  (their heuristic: cosine 0.2 + WER 30 + Flan-T5 perplexity 200 -- a different"
      "\n   feature set from ours, but the same kind of instrument)")

J0 = JUDGES[0]
print(f"\nwhere the taxonomy and {J0} disagree, by taxonomy category "
      f"({HEAD[0]} s / ts {HEAD[1]})\n")
hdr = f"{'taxonomy category':>18} {'n':>6} {'llm halluc':>11} {'tax halluc':>11} {'disagree':>9}"
print(hdr + "\n" + "-" * len(hdr))
for cat in ["faithful", "errorful", "truncated", "empty", "degenerate", "runaway", "untethered"]:
    rs = [r for m in MODELS for r in BY[(m, *HEAD)] if r["taxonomy"] == cat]
    if not rs:
        continue
    dis = sum(r["taxonomy_halluc"] != r[COL[J0]] for r in rs)
    print(f"{cat:>18} {len(rs):>6} {sum(r[COL[J0]] for r in rs):>11}"
          f" {sum(r['taxonomy_halluc'] for r in rs):>11} {dis:>9}")

print("\n`degenerate` is the row to read first: Atwany et al. classify repetition as"
      "\nOscillation Error -- a NON-hallucination -- so the judge should disagree with our"
      "\ntaxonomy there by construction. That is a definitional split, not a detector failure.")

## 9. Write the results

Git-safe by construction: every column is an identifier or a category name drawn from a fixed label
set. There is no free text to leak, so this file needs no Drive-only companion — the same property
`halluc_taxonomy.csv` has.

The provenance records the licensing decision explicitly. A later reader should be able to see that
transmitting reference text was chosen deliberately, with the grid size it applied to, rather than
having to infer it from the fact that the notebook exists.

In [ ]:
SAFE_FIELDS = (["model", "path", "speaker", "region", "offset_s", "timestamps",
                "taxonomy", "taxonomy_halluc"]
               + [f"verdict_{j}" for j in JUDGES] + [f"halluc_{j}" for j in JUDGES])
FORBIDDEN = {"text", "reference", "hypothesis", "ref", "hyp", "transcript"}
IDENT = ({"model", "path", "speaker", "region", "timestamps", "taxonomy"}
         | {f"verdict_{j}" for j in JUDGES})

assert not (set(SAFE_FIELDS) & FORBIDDEN), "a text-bearing column leaked into SAFE_FIELDS"
assert set(rows[0]) == set(SAFE_FIELDS), set(rows[0]) ^ set(SAFE_FIELDS)
for row in rows:
    for k, v in row.items():
        assert k in IDENT or " " not in str(v), (k, v)
    for j in JUDGES:
        assert row[f"verdict_{j}"] in LABELS, row[f"verdict_{j}"]

with open(OUT_CSV, "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=SAFE_FIELDS)
    wr.writeheader()
    wr.writerows(rows)

PROVENANCE = {
    "experiment": "LLM-based hallucination classification (Atwany et al. 2025, ACL Findings) "
                  "over experiment A's hypotheses, as an independent check on the text taxonomy",
    "method": {
        "prompt": "Atwany et al. Figure 5 (coarse-grained), transcribed verbatim",
        "grain": GRAIN, "labels": LABELS,
        "judges": JUDGES, "temperature": TEMPERATURE,
        "replication": "same model family (gpt-4o-mini), same prompt, same greedy decoding "
                       "as the source paper",
        "deviations": [
            "response_format json_schema (strict) replaces the prompt's 'produce only the "
            "classification' instruction, removing the parse step and foreclosing both a "
            "preamble and an off-vocabulary label",
        ],
        "batch_api": True, "max_tokens": MAX_TOKENS,
    },
    "licensing": {
        "decision": "full-grid submission chosen deliberately by the author",
        "consequence": "all 1000 TIMIT (LDC93S1) reference transcripts and 20000 hypotheses "
                       "were transmitted to the OpenAI API",
        "note": "every other notebook in this project keeps reference text off the wire; this "
                "one does not, and the trade was made knowingly rather than overlooked",
    },
    "derived_from": {"inputs": INPUT_DIGESTS, "reference_digest": rebuilt,
                     "batches": submitted},
    "packages": {"python": platform.python_version(), "openai": openai.__version__,
                 "tiktoken": tiktoken.__version__, "numpy": np.__version__},
    "grid": {"models": MODELS, "conditions": [f"{o}s/ts-{t}" for o, t in CONDS],
             "clips_per_cell": 1000, "classifications": len(rows) * len(JUDGES)},
    "usage": {f"{j}|{k}": v for (j, k), v in usage_tot.items()},
    "agreement": AGREE,
    "summary": SUMMARY,
    "outputs": {"llm_verdict_per_utterance.csv": {"rows": len(rows), "fields": SAFE_FIELDS,
                                                  "git_safe": True}},
}
with open(PROV_JSON, "w") as f:
    json.dump(PROVENANCE, f, indent=1)

_back = list(csv.DictReader(open(OUT_CSV, newline="")))
assert len(_back) == len(rows) and set(_back[0]) == set(SAFE_FIELDS)

print(f"per-utterance -> {OUT_CSV}   ({len(rows)} rows, labels only; safe to commit)")
print(f"provenance    -> {PROV_JSON}   (records the licensing decision)")
print("\nchecked: no text-bearing column, every verdict drawn from the fixed label set")

## 10. Figure

The question the figure has to answer is whether two independent methods see the same shape across
scale — so both go on one axis, per arm, and the reader compares slopes rather than levels. Levels
will differ (the methods disagree about repetition by construction); a shared slope is the claim.

Two panels for the two timestamp arms, because the flat off-arm is what makes the on-arm meaningful.
Vector PDF at `pdf.fonttype = 42`, downloaded to local disk alongside the PNG.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e5e2"
CCAT   = ["#2a78d6", "#7a9a3b"]                       # fixed order, validated against SURFACE
SERIES = {"taxonomy": ("#eb6834", "text taxonomy")}
for n, j in enumerate(JUDGES):
    SERIES[j] = (CCAT[n % len(CCAT)], f"LLM HER ({j})")

x = np.arange(len(MODELS))
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6), dpi=160, sharey=True)
fig.patch.set_facecolor(SURFACE)

for ax, arm in zip(axes, ("on", "off")):
    ax.set_facecolor(SURFACE)
    ax.grid(True, axis="y", color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    for src, (colour, label) in SERIES.items():
        s   = [SUMMARY[f"{src}|{m}|{HEAD[0]}|{arm}"] for m in MODELS]
        mid = [100 * v["her"] for v in s]
        err = [[100 * (v["her"] - v["wilson_lo"]) for v in s],
               [100 * (v["wilson_hi"] - v["her"]) for v in s]]
        ax.errorbar(x, mid, yerr=err, fmt="o", color=colour, markersize=7, linewidth=2,
                    markeredgecolor=SURFACE, markeredgewidth=2, capsize=4, zorder=3,
                    label=label if arm == "off" else None)
        ax.plot(x, mid, color=colour, linewidth=2, zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{m}\n{PARAMS[m]}" for m in MODELS])
    ax.set_xlim(-0.5, len(MODELS) - 0.5)
    ax.set_title(f"timestamps {arm}", fontsize=12, color=INK, pad=10, loc="left")
    ax.tick_params(colors=INK2, labelsize=10, length=0)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color(GRID); ax.spines[sp].set_linewidth(1)

axes[0].set_ylabel(f"flagged at {HEAD[0]} s (%)", fontsize=11, color=INK2)
leg = axes[1].legend(frameon=False, fontsize=9.5, loc="upper right", title="detector")
leg.get_title().set_color(INK2); leg.get_title().set_fontsize(9)
for t in leg.get_texts():
    t.set_color(INK2)
fig.suptitle("Two independent detectors, one scale trend "
             f"(utterances at {HEAD[0]} s, n = 1000 per point)",
             fontsize=12.5, color=INK, x=0.007, ha="left", y=1.02)
fig.tight_layout()

for ext in ("pdf", "png"):
    fig.savefig(f"llm_vs_taxonomy.{ext}", bbox_inches="tight", facecolor=SURFACE)
    fig.savefig(os.path.join(DRIVE_ROOT, f"llm_vs_taxonomy.{ext}"),
                bbox_inches="tight", facecolor=SURFACE)
print("wrote llm_vs_taxonomy.pdf and .png (+ copies on Drive)")
plt.show()

from google.colab import files
files.download("llm_vs_taxonomy.pdf")
files.download("llm_vs_taxonomy.png")

## 11. Standalone reload

No API key, no TIMIT, no Drive-only file, no earlier cell. Reads the committed
`llm_verdict_per_utterance.csv` and rebuilds both the HER table and the agreement figures — which
proves the git-safe file alone carries the whole result, and that nothing here depends on re-running
a paid API call.

In [ ]:
# --- standalone: run this alone in a fresh CPU runtime -------------------------------
import csv, collections, itertools, math, os
import numpy as np

PATH = "/content/drive/MyDrive/NAACL/llm_verdict_per_utterance.csv"   # or a local download
rows = list(csv.DictReader(open(PATH, newline="")))

MODELS_ = ["tiny", "base", "small", "medium", "large-v3"]
CONDS_  = [(5, "on"), (5, "off"), (25, "on"), (25, "off")]
JUDGES_ = [c[len("halluc_"):] for c in rows[0]
           if c.startswith("halluc_") and c != "halluc_taxonomy"]

by = collections.defaultdict(list)
for r in rows:
    by[(r["model"], int(r["offset_s"]), r["timestamps"])].append(r)


def wilson_(k, n, z=1.959963985):
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


print(f"reloaded {len(rows)} rows | judges: {', '.join(JUDGES_)}\n")
for src in ["taxonomy"] + JUDGES_:
    col = "taxonomy_halluc" if src == "taxonomy" else f"halluc_{src}"
    print(f"{src}")
    print(f"{'model':>9} " + " ".join(f"{f'{o} s / ts {t}':>21}" for o, t in CONDS_))
    for m in MODELS_:
        out = []
        for o, t in CONDS_:
            rs = by[(m, o, t)]
            p, lo, hi = wilson_(sum(int(r[col]) for r in rs), len(rs))
            out.append(f"{100*p:5.1f} [{100*lo:4.1f},{100*hi:4.1f}]")
        print(f"{m:>9} " + " ".join(f"{c:>21}" for c in out))
    print()

print("agreement (raw / kappa / jaccard)")
cols = ["taxonomy_halluc"] + [f"halluc_{j}" for j in JUDGES_]
for ca, cb in itertools.combinations(cols, 2):
    a = np.array([int(r[ca]) for r in rows])
    b = np.array([int(r[cb]) for r in rows])
    po = float((a == b).mean())
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    kap = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    jac = (a & b).sum() / max(1, (a | b).sum())
    print(f"  {ca:>22} vs {cb:<22} {po:.3f} / {kap:.3f} / {jac:.3f}")